##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [3]:
!pip install -U -q "google-genai>=2.9.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [4]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [5]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [6]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is **Paris**.

In [7]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdCczdrYXJTVEd0Zlo5dE1QNnRxTHVRURIXQnM3a2FyU1RHdGZaOXRNUDZ0cUx1UVE
Environment ID: 1880c1d73ac2088d4951485050d535de


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [8]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have created `knowledge.md` and recorded your information:

- **Name:** Alice
- **Role:** Software Engineer

In [9]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** Based on your profile in `knowledge.md`:

- **Name:** Alice
- **Profession/Role:** Software Engineer

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [10]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the Python script to compute the first 20 Fibonacci numbers, followed by the execution output.

### Python Script (`fibonacci.py`)

```python
def fibonacci(n: int) -> list[int]:
    """Generate the first n Fibonacci numbers (starting with 0 and 1)."""
    if n <= 0:
        return []
    if n == 1:
        return [0]
    
    fib = [0, 1]
    for _ in range(2, n):
        fib.append(fib[-1] + fib[-2])
    return fib

if __name__ == "__main__":
    n = 20
    fib_numbers = fibonacci(n)
    print(f"First {n} Fibonacci numbers:")
    print(fib_numbers)
    print("\nIndexed list:")
    for i, val in enumerate(fib_numbers):
        print(f"F({i}) = {val}")
```

---

### Execution Output

```text
First 20 Fibonacci numbers:
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]

Indexed list:
F(0) = 0
F(1) = 1
F(2) = 1
F(3) = 2
F(4) = 3
F(5) = 5
F(6) = 8
F(7) = 13
F(8) = 21
F(9) = 34
F(10) = 55
F(11) = 89
F(12) = 144
F(13) = 233
F(14) = 377
F(15) = 610
F(16) = 987
F(17) = 1597
F(18) = 2584
F(19) = 4181
```

*(Note: If starting the sequence from $F_1 = 1, F_2 = 1$, the 20th Fibonacci number is $F_{20} = 6765$.)*

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [11]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [function_call] ---
  Tool: list_dir
  Args: {'DirectoryPath': '/'}

--- Step 2 [function_result] ---
  Tool: list_dir

--- Step 3 [function_call] ---
  Tool: list_dir
  Args: {'DirectoryPath': '/workspace'}

--- Step 4 [function_result] ---
  Tool: list_dir

--- Step 5 [thought] ---

--- Step 6 [function_call] ---
  Tool: write_to_file
  Args: {'CodeContent': 'def fibonacci(n: int) -> list[int]:\n    """Generate the first n Fibonacci numbers (starting with 0 and 1)."""\n    if n <= 0:\n        return []\n    if n == 1:\n        return [0]\n    \n    fib = [0, 1]\n    for _ in range(2, n):\n        fib.append(fib[-1] + fib[-2])\n    return

--- Step 7 [function_result] ---
  Tool: write_to_file

--- Step 8 [code_execution_call] ---

--- Step 9 [code_execution_result] ---

--- Step 10 [thought] ---

--- Step 11 [model_output] ---
  Text: Here is the Python script to compute the first 20 Fibonacci numbers, followed by the execution output.

### Python

### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [12]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

Here are the top three news stories about Google this week, along with brief summaries:

---

### 1. Alphabet Reports Blockbuster Q3 2026 Earnings & Milestone Growth
Alphabet announced its **Q3 2026 financial results**, highlighting significant momentum across its AI and enterprise segments:
* **Financial Milestones:** Revenue grew **25% year-over-year**, bringing Alphabet to a historic **$500 billion annual revenue run rate**. Google Cloud saw rapid acceleration, with revenues climbing **103% YoY** fueled by surging demand for AI infrastructure and Google's custom Tensor Processing Units (TPUs).
* **Consumer Milestones:** CEO Sundar Pichai announced that the **Gemini app reached 1 billion users**, making it the fastest-growing consumer product in Google’s history. In addition, YouTube Music and Premium crossed **150 million subscribers** globally.
* Read more: [Google Blog – Q3 2026 Earnings Remarks](https://blog.google/company-news/inside-google/message-ceo/alphabet-earnings-q3-2026/)

---

### 2. DeepMind Unveils Next-Generation Frontier Model: Gemini 4 Carbon
Google expanded its Gemini 4 lineup with the introduction of **Gemini 4 Carbon**, its most powerful frontier model to date:
* **Focus on Cyber Defense & Reasoning:** Carbon is optimized for long-horizon agentic workflows, software engineering, and cybersecurity defense. Evaluated across major security benchmarks, it was shown to detect and mitigate complex vulnerabilities with unprecedented speed.
* **Benchmark Leadership:** Carbon claimed the #1 spot on the Artificial Analysis Intelligence Index v5 (scoring 59), outperforming Gemini 4 Argon.
* **Controlled Rollout:** Given its advanced dual-use capabilities, Google is rolling Carbon out first to vetted cybersecurity defenders, government partners, and select enterprise teams before wider developer availability.
* Read more: [Google Blog – Gemini 4 Carbon: Our Next Frontier Model](https://blog.google/innovation-and-ai/models-and-research/gemini-models/introducing-gemini-4-carbon/)

---

### 3. Google DeepMind Announces Genie 4 Interactive World Model
Google DeepMind unveiled **Genie 4**, a breakthrough foundation model for real-time interactive simulation:
* **Explorable Real-Time Environments:** Unlike conventional video generation models that render static clips, Genie 4 generates dynamic, interactive environments frame-by-frame that users can steer and manipulate using prompt inputs and keyboard navigation.
* **Native Audio Breakthrough:** Genie 4 is DeepMind's first world model to generate synchronized **native audio**—including ambient soundscapes, background acoustics, and reactive dialogue—directly in real time. 
* **Target Applications:** Beyond gaming and content creation, Google positioned Genie 4 as a vital environment for training multimodal AI agents and embodied robotics.
* Read more: [Google Blog – Introducing Genie 4](https://blog.google/innovation-and-ai/models-and-research/google-deepmind/introducing-genie-4/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [13]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created the Python file `analysis.py`.

### Script Code (`analysis.py`)

```python
import random
import statistics

def main():
    # Generate 50 random numbers (uniformly distributed between 1 and 100)
    numbers = [random.uniform(1, 100) for _ in range(50)]

    # Compute statistics
    mean_val = statistics.mean(numbers)
    median_val = statistics.median(numbers)
    stdev_val = statistics.stdev(numbers)

    # Print the results
    print("Generated 50 Random Numbers:")
    for i in range(0, len(numbers), 10):
        print("  " + ", ".join(f"{n:6.2f}" for n in numbers[i:i+10]))

    print("\nStatistical Analysis:")
    print(f"  Count:              {len(numbers)}")
    print(f"  Mean:               {mean_val:.4f}")
    print(f"  Median:             {median_val:.4f}")
    print(f"  Standard Deviation: {stdev_val:.4f}")

if __name__ == "__main__":
    main()
```

---

### Execution Output

Running `python3 analysis.py`:

```
Generated 50 Random Numbers:
   82.37,  80.94,  11.50,  99.78,  72.82,  15.59,  40.31,  98.44,  93.26,  69.18
   86.91,  14.88,  48.39,  87.88,  30.29,  68.41,  85.83,   3.60,  24.12,  73.95
   92.85,  32.96,  24.02,  50.08,  93.46,  75.40,  35.17,  43.76,  91.81,  38.66
   74.69,  12.75,  51.59,  89.93,  45.45,  25.82,  49.17,  33.91,  50.20,  21.42
   58.91,  25.23,  86.13,  51.14,  30.52,  57.57,  26.28,  49.84,  67.95,  63.29

Statistical Analysis:
  Count:              50
  Mean:               54.7679
  Median:             50.6665
  Standard Deviation: 27.4421
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [14]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on the analysis of `data.csv`, **Charlie** scored the highest.

### Score Summary:
- **Charlie**: **95** (Age: 42, City: Berlin)
- **Alice**: 92 (Age: 28, City: Paris)
- **Eve**: 91 (Age: 26, City: Sydney)
- **Diana**: 88 (Age: 31, City: Tokyo)
- **Bob**: 87 (Age: 35, City: London)

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [15]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

/tmp/ipykernel_5672/1261841246.py:2: UserWarning: Agents usage is experimental and may change in future versions.
  my_agent = client.agents.create(


✓ Agent created: my-data-analyst-dba8d77e


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [16]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the Python code using **pandas** to generate the 100-record sales dataset, analyze the top 5 products by total revenue, and produce the visualization.

---

### 1. Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Define product catalog with unit price ranges
products_catalog = {
    'Laptop': (800, 1500),
    'Smartphone': (500, 1000),
    'Smart TV': (400, 900),
    'Tablet': (300, 700),
    'Gaming Console': (350, 500),
    'Monitor': (150, 400),
    'Smartwatch': (150, 350),
    'Headphones': (80, 250),
    'Mechanical Keyboard': (50, 150),
    'Wireless Speaker': (40, 120)
}

regions = ['North', 'South', 'East', 'West']

# 1. Generate 100 sales records
n_records = 100
product_list = list(products_catalog.keys())
sampled_products = np.random.choice(product_list, size=n_records)
sampled_regions = np.random.choice(regions, size=n_records)
sampled_quantities = np.random.randint(1, 11, size=n_records)

# Calculate revenue based on unit price * quantity
revenues = []
for prod, qty in zip(sampled_products, sampled_quantities):
    low, high = products_catalog[prod]
    unit_price = np.round(np.random.uniform(low, high), 2)
    revenue = np.round(unit_price * qty, 2)
    revenues.append(revenue)

# Create DataFrame with required columns: product, region, revenue, quantity
df = pd.DataFrame({
    'product': sampled_products,
    'region': sampled_regions,
    'revenue': revenues,
    'quantity': sampled_quantities
})

# Save dataset to CSV
df.to_csv('sales_data.csv', index=False)

# 2. Analyze Top 5 Products by Total Revenue
top_5_analysis = df.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    order_count=('revenue', 'count'),
    avg_order_revenue=('revenue', 'mean')
).sort_values(by='total_revenue', ascending=False)

top_5_analysis['revenue_share_pct'] = (top_5_analysis['total_revenue'] / df['revenue'].sum()) * 100
top_5 = top_5_analysis.head(5)

# 3. Regional breakdown for top 5 products
regional_breakdown = df[df['product'].isin(top_5.index)].pivot_table(
    index='product',
    columns='region',
    values='revenue',
    aggfunc='sum',
    fill_value=0
).loc[top_5.index]

# 4. Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Total Revenue Bar Chart
colors = sns.color_palette("Blues_r", n_colors=5)
bars = ax1.bar(top_5.index, top_5['total_revenue'], color=colors, edgecolor='black', linewidth=0.8)
ax1.set_title('Top 5 Products by Total Revenue', fontsize=14, fontweight='bold', pad=15)
ax1.set_ylabel('Total Revenue ($)', fontsize=12)
ax1.set_xlabel('Product', fontsize=12)
ax1.tick_params(axis='x', rotation=30)
ax1.yaxis.set_major_formatter('${x:,.0f}')

for bar in bars:
    height = bar.get_height()
    ax1.annotate(f'${height:,.0f}',
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 4),
                 textcoords="offset points",
                 ha='center', va='bottom', fontsize=10, fontweight='bold')

# Subplot 2: Regional Revenue Breakdown (Stacked Bar Chart)
regional_breakdown.plot(kind='bar', stacked=True, ax=ax2, colormap='Spectral', edgecolor='black', linewidth=0.8)
ax2.set_title('Regional Revenue Breakdown for Top 5 Products', fontsize=14, fontweight='bold', pad=15)
ax2.set_ylabel('Revenue ($)', fontsize=12)
ax2.set_xlabel('Product', fontsize=12)
ax2.tick_params(axis='x', rotation=30)
ax2.yaxis.set_major_formatter('${x:,.0f}')
ax2.legend(title='Region', frameon=True)

plt.tight_layout()
plt.savefig('top_5_products_revenue.png', dpi=300)
plt.close()
```

---

### 2. Dataset Sample (First 10 Records)

| # | Product | Region | Revenue ($) | Quantity |
|---|---|---|---|---|
| 0 | Smartwatch | West | \$224.46 | 1 |
| 1 | Tablet | South | \$2,028.15 | 3 |
| 2 | Headphones | North | \$2,455.20 | 10 |
| 3 | Gaming Console | West | \$3,140.72 | 8 |
| 4 | Smartwatch | East | \$1,266.42 | 6 |
| 5 | Wireless Speaker | East | \$630.80 | 8 |
| 6 | Smart TV | South | \$5,617.89 | 9 |
| 7 | Smartwatch | West | \$1,395.56 | 4 |
| 8 | Headphones | North | \$109.91 | 1 |
| 9 | Gaming Console | East | \$352.71 | 1 |

- **Total Records:** 100
- **Overall Revenue:** $193,266.40
- **Total Units Sold:** 496

---

### 3. Top 5 Products by Total Revenue

| Rank | Product | Total Revenue | Units Sold | Orders | Avg. Order Revenue | Revenue Share |
|:---:|---|:---:|:---:|:---:|:---:|:---:|
| 1 | **Laptop** | **\$38,083.92** | 34 | 7 | \$5,440.56 | 19.71% |
| 2 | **Tablet** | **\$33,414.09** | 60 | 9 | \$3,712.68 | 17.29% |
| 3 | **Smartphone** | **\$32,262.15** | 48 | 10 | \$3,226.22 | 16.69% |
| 4 | **Smart TV** | **\$27,096.78** | 46 | 9 | \$3,010.75 | 14.02% |
| 5 | **Gaming Console** | **\$18,777.90** | 45 | 10 | \$1,877.79 | 9.72% |

---

### 4. Regional Breakdown for Top 5 Products

| Product | East | North | South | West |
|---|:---:|:---:|:---:|:---:|
| **Laptop** | \$14,932.75 | \$15,052.85 | \$8,098.32 | \$0.00 |
| **Tablet** | \$7,894.18 | \$11,126.56 | \$2,028.15 | \$12,365.20 |
| **Smartphone** | \$8,611.59 | \$5,359.14 | \$2,982.41 | \$15,309.01 |
| **Smart TV** | \$13,205.72 | \$813.93 | \$8,570.49 | \$4,506.64 |
| **Gaming Console** | \$2,770.35 | \$3,339.64 | \$6,873.21 | \$5,794.70 |

---

### 5. Key Findings

1. **Revenue Concentration:** The top 5 products account for **\$149,634.84** or **77.43%** of all revenue generated across the 100 sales transactions.
2. **Top Performer (Laptop):** While **Laptop** had the lowest transaction volume among the top 5 (7 orders, 34 units), its high price point drove the highest total revenue (\$38,083.92) and the highest average order value (\$5,440.56).
3. **Volume vs. Value (Tablet & Smartphone):** **Tablets** and **Smartphones** drove steady volume (60 and 48 units sold respectively), contributing over 33% of aggregate revenue combined.
4. **Geographic Distribution:**
   - **Laptop** sales were strongest in the **East** and **North** regions.
   - **Smartphone** sales performed strongest in the **West** (\$15,309.01).
   - **Smart TV** performed best in the **East** (\$13,205.72).

### Generated Files
- `sales_data.csv` — Contains the raw 100-record dataset.
- `top_5_products_revenue.png` — Visualization including the total revenue comparison and regional breakdown charts.

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [17]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-dba8d77e


In [18]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

I have three specialized skills configured to design, build, and deploy applications using the Gemini ecosystem:

---

### 1. **Gemini API Development (`gemini-api-dev`)**
Covers the core Gemini API across Python (`google-genai`) and JavaScript/TypeScript (`@google/genai`) using the modern Interactions API and current model family:

* **Text & Reasoning**: Build multi-turn chat, complex reasoning, and coding applications with models like `gemini-4-argon`, `gemini-3.8-flash`, and lightweight options like `gemini-3.5-flash-lite` or `gemma-4`.
* **Multimodal Understanding**: Analyze text, high-resolution images, long videos, audio, and documents in a unified context.
* **Image Generation & Editing**: Generate and conversationally edit images with multi-image reference fusion (up to 14 references) using Nano Banana models (`gemini-nano-banana-2.1`, `gemini-3-pro-image`, `gemini-3.1-flash-lite-image`).
* **Speech & Audio**:
  * **Text-to-Speech (TTS)**: Multi-speaker dialogue, vocal tags, custom Voice Design, and Voice Replication via `gemini-3.8-flash-tts` and `gemini-3.8-flash-lite-tts`.
  * **Speech-to-Text**: Fast transcription with smart or verbatim modes via `gemini-3.5-transcribe`.
* **Autonomous Agents & Deep Research**: Provision sandboxed managed agents (`antigravity-preview-09-2026`) or run exhaustive automated research with `deep-research-preview-04-2026`.
* **Structured Output & Function Calling**: Tool use, JSON schema enforcement, and background asynchronous jobs.
* **API Migration**: Upgrade legacy codebases from deprecated models (`gemini-1.5`, `gemini-2.0`, `gemini-2.5`) and deprecated SDKs (`google-generativeai`).

---

### 2. **Real-Time Bidirectional Streaming (`gemini-live-api-dev`)**
Enables ultra-low-latency voice, video, and text interactions over WebSockets:

* **Voice & Conversational Agents**: Build mic-to-speaker voice agents with `gemini-3.8-live`.
* **Extended Thinking & Fillers**: Use `gemini-3.8-live-extended-thinking` to perform background reasoning and execute asynchronous non-blocking tools while maintaining continuous, spoken conversational fillers.
* **Live Video Streaming**: Send camera or screen capture frames alongside real-time audio.
* **Live Transcription & Translation**:
  * Real-time speech-to-text with interim and finalized transcripts (`gemini-3.5-transcribe-live`).
  * Live speech-to-speech translation across 70+ languages (`gemini-3.5-live-translate-preview`).
* **Session & Audio Controls**: Server-side VAD, client-side Hybrid VAD, push-to-talk, session context compression, resumption, and ephemeral token generation for secure client connections.
* **Ecosystem Integrations**: Architect pipelines with frameworks like LiveKit, Pipecat, Fishjam, and Firebase AI SDK.

---

### 3. **Generative Video Creation & Editing (`gemini-omni-flash-api`)**
Specialized in generative video synthesis and video editing using `gemini-omni-1.1-flash`:

* **Text-to-Video**: Generate video from detailed descriptions in resolutions ranging from `360p`, `720p`, and `1080p` up to `4K`, in `16:9` or `9:16` aspect ratios.
* **First-Frame & Frame Transitions**:
  * Animate a static starting image into video.
  * Interpolate smoothly between a start frame and an end frame.
* **Video Extensions**: Sequentially extend existing clips up to 40 seconds total.
* **Video Editing & Stylization**: Modify existing footage (style transfers, object modifications, inpainting/outpainting) with image/video reference conditioning.
* **Media Pipelines**: Automated media preparation, audio-stripping for generative re-scoring, and `ffmpeg` pre-processing.

---

Whether you want to generate code, build a real-time voice agent, edit videos, or design a multi-agent workflow, let me know what you would like to build!

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [20]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [22]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id=f"my-data-analyst-{UNIQUE_SUFFIX}")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- my-data-analyst-dba8d77e
- my-forked-agent
- my-gemini-api-agent-dba8d77e

Agent details for my-data-analyst-dba8d77e:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [23]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst
✓ Deleted my-forked-agent
✓ Deleted my-gemini-api-agent


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [24]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
[step.stop]
[step.start]
Beneath the wide and wandering sky,  
The sapphire rollers crest and sigh.  
A heartbeat carved in salt and foam,  
Where restless tides forever roam.  

From sunlit spray to silent floor,  
It whispers secrets to the shore—  
An endless cradle, wild and deep,  
Where ancient shadows wake and sleep.[step.stop]
[interaction.completed]


--- Collected 4 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [25]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

To list the available models from the Gemini API and inspect the first 3, you can use `curl` to query the `v1beta/models` endpoint (piped to `jq` to slice the first 3 entries):

```bash
curl -s "https://generativelanguage.googleapis.com/v1beta/models" | jq '.models[:3]'
```

### Result

```json
[
  {
    "name": "models/gemini-2.5-flash",
    "version": "001",
    "displayName": "Gemini 2.5 Flash",
    "description": "Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-pro",
    "version": "2.5",
    "displayName": "Gemini 2.5 Pro",
    "description": "Stable release (June 17th, 2025) of Gemini 2.5 Pro",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-flash-preview-tts",
    "version": "gemini-2.5-flash-exp-tts-2025-05-19",
    "displayName": "Gemini 2.5 Flash Preview TTS",
    "description": "Gemini 2.5 Flash Preview TTS",
    "inputTokenLimit": 8192,
    "outputTokenLimit": 16384,
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [26]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: e0995ac62dcbd1fe4364fa5e448cb444


The `project` directory has been created along with the requested files.

### Created Files in `project/`:
- `README.md`: Contains basic project documentation and instructions on running the script.
- `hello.py`: A Python script that prints `"Hello, world!"`.

In [27]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./workspace (0 bytes)
  ./workspace/project (0 bytes)
  ./project (0 bytes)
  ./project/README.md (144 bytes)
  ./project/hello.py (79 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations